# Comms / Dropout / Restart Analysis

Diagnoses of driver communications related issues covering:
* Heatbeat lag (checked every 0.3s, from outside event loop, for a delay of >0.2s)
* Event loop lag (check every 0.5s, from inside watchdog 500ms event loop, for a delay of >0.1s)
* Position Update lag (518 messages, short delays, long delays with comms restart)
* Telemetry gaps (KFLOG and PIDLOG gaps)
* CPU Load, Memory Load, Network Errors
* Socket connection issues (hard socket disconnects)
* manual client recovery
  


In [1]:
import os, sys
import re
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import importlib
import analyse_helpers
importlib.reload(analyse_helpers)
from analyse_helpers import (
    resolve_log_files, load_kf_pid, load_vitals, load_high_cpu_events,
    load_connection_events, reconstruct_outages,
)

ARCSEC = 3600  # deg -> arcsec, same convention as analyse_kf_pid.ipynb

def wrap_deg(x):
    """Wrap a degree value/difference into (-180, 180] -- see analyse_kf_pid.ipynb's own copy."""
    return (x + 180) % 360 - 180


# Load data

In [2]:
# -- Choose correct log path (last set log_filenames is what is used) --------------------------------
# Single file:            'alpaca.log'                     (relative to LOG_DIR below)
# All rotated files:      'alpaca.log*'                    (glob -- handles a multi-hour run spanning several files)
# Explicit file list:     ['alpaca.log.2', 'alpaca.log.1', 'alpaca.log']
LOG_DIR = '../logs/archive'   # base directory log_filenames below are resolved against

log_filenames = ['alpaca.soak_nopec_Beta4.3_08_31a*.log']                           # the overnight capture with the real dropout this notebook was built against
log_filenames = ['alpaca.greg_Beta3.1_08_02_comms_problems.log']                    # comms problems with lots of lag
log_filenames = ['alpaca.jdm_Beta3.1_08_02_sg_peclog.log']                          # sync guiding      with logs(pec)
log_filenames = ['alpaca.jdm_Beta3.1_08_03_sg_peclog.log']                          # sync guiding      with logs(pec)
log_filenames = ['alpaca.jdm_Beta4.3_08_30_kfposlog_d*.log']                        # tracking          with logs(kf,pos) and socket force closed from Polaris  
log_filenames = ['alpaca.soak_Beta4.3_08_31_sg_kfposlog_Dec72_504m_a*.log']         # sync guiding      with logs(kf,pos) on 47 Tuc, socket force closed from Polaris at 00:47 am  
log_filenames = ['alpaca.soak_Beta4.3_08_30_sg_Dec23N_120min_b*.log']               # sync guiding      with logs(none), northern hemishphere NGC6823

log_filenames = ['alpaca.soak_Beta4.3_08_30_sg_peckfposlog_72min_a*.log']           # sync guiding, pec with logs(pec,kf,pos), northern hemishphere NGC6823
log_filenames = ['alpaca.soak_Beta4.4_09_01_sg_sglog_Dec10_a*.log']                 # sync guiding,     with logs(sg), Eagle Nebula


resolved_files = resolve_log_files(log_filenames, log_dir=LOG_DIR)
print(f"Loading {len(resolved_files)} file(s) from LOG_DIR={LOG_DIR!r}:")
for p in resolved_files:
    print(f"  {p}  ({os.path.getsize(p)/1e6:.1f} MB)")

kf_df, pid_df = load_kf_pid(log_filenames, log_dir=LOG_DIR)
print(f"\nKF  samples: {len(kf_df)}"  + (f"  ({kf_df.t_sec.iloc[-1]/60:.1f} min span)"  if len(kf_df)  else ""))
print(f"PID samples: {len(pid_df)}" + (f"  ({pid_df.t_sec.iloc[-1]/60:.1f} min span)" if len(pid_df) else ""))


Loading 1 file(s) from LOG_DIR='../logs/archive':
  ../logs/archive/alpaca.soak_Beta4.4_09_01_sg_sglog_Dec10_a1.log  (0.4 MB)

KF  samples: 0
PID samples: 0


# Connection lifecycle events (restarts, disconnects, reconnect attempts, config toggles)

Every driver process (re)start, socket disconnect, reconnect attempt (both the
"fresh-connect-failed" and "established-connection-died" flavors -- see
`load_connection_events()`'s docstring), successful re-init, WiFi lifecycle event (netsh
join failures *and* BLE-mediated `bleEnableWifi` failures), and the client REST actions
that drive or observe reconnection, in one chronologically sorted table plus a timeline
chart below it.

Every failure-carrying row (`disconnect`/`reconnect_error`/`connect_attempt_failed`/
`wifi_join_failed`/`cmd_timeout`/`send_error`/`ble_wifi_failed`/`ble_error`) is additionally
run through `classify_connection_error()` -- a taxonomy built by grepping every unique
error string across all 79 captures in `logs/archive` and cross-referencing
`driver/polaris.py`'s own `_format_connection_error()` (the sole place that turns a raw
exception into this log text) plus `docs/troubleshooting.md`'s C0-C3 sections. It adds:
* `error_code` -- a short stable id, e.g. `WSAECONNRESET`, `SEM_TIMEOUT`, `WATCHDOG_NO_TELEMETRY`
* `error_meaning` -- a one-line explanation, with a troubleshooting-doc section reference where one exists
* `error_category` -- a coarser grouping (`peer_reset`, `link_degraded`, `unreachable`, `watchdog`, `wifi_join`, `ble`, ...) used to color the chart below

An unrecognized Windows/OS error code still gets a `WINERR_<n>`/`ERRNO_<n>` code rather
than vanishing, so a new failure mode this taxonomy hasn't seen yet is still visible and
groupable.

The chart plots every event in this table (not only classified failures: its y-axis is
`error_code` where one exists, and the bare `kind` otherwise -- `driver_start`,
`init_start`/`init_done`, `wifi_interface`, `config_update`, `client_action`), **plus**
the three `load_vitals()` lag warnings (`Heartbeat lag detected`, `Event loop lag
detected`, `518 lag detected`) on the same timeline -- these are early-warning
symptoms that often precede a disconnect (see the Notes worked example below, where
escalating heartbeat/event-loop lag shows up ~10s before the WSAECONNRESET that
triggered that outage), so seeing them lined up against the failures around them is often
the fastest way to tell "CPU-starved event loop" from "genuine network/Polaris-side
problem" apart. Only `sync_observed` is left off the chart/table (high-volume, mostly
just context -- still present in `connection_events_df` itself, and its count is printed
above the table).

`config_update` rows carry the REST call's own `Parameters` as `param_*` columns -- watch
for `param_log_position`/`param_log_pec`/`param_advanced_pec` here before treating a gap
elsewhere as pure data loss; it might just be a logging toggle.

In [3]:
connection_events_df = load_connection_events(log_filenames, log_dir=LOG_DIR)
# Loaded independently here (not reused from the later "Driver vitals events" cell) so
# this cell stays runnable on its own regardless of notebook execution order.
vitals_for_chart_df = load_vitals(log_filenames, log_dir=LOG_DIR)

ERROR_CATEGORY_COLORS = {
    'watchdog': 'red', 'peer_reset': 'orangered', 'closed_clean': 'gold',
    'adapter_reset': 'deeppink', 'local_abort': 'magenta', 'refused': 'purple',
    'link_degraded': 'yellow', 'unreachable': 'tomato', 'connect_timeout': 'royalblue',
    'wifi_join': 'deepskyblue', 'ble': 'mediumseagreen', 'config_state': 'gray', 'other': 'white',
}
LIFECYCLE_KIND_COLORS = {
    'driver_start': 'cyan', 'init_start': 'palegreen', 'init_done': 'seagreen',
    'wifi_interface': 'turquoise', 'config_update': 'khaki', 'client_action': 'lightgray',
}
LIFECYCLE_KIND_ORDER = ['driver_start', 'init_start', 'init_done', 'wifi_interface',
                         'config_update', 'client_action']  # top-to-bottom reading order on the chart
# Colors deliberately distinct from ERROR_CATEGORY_COLORS/LIFECYCLE_KIND_COLORS above --
# these three share this one chart's legend with those, unlike the Comms Anomaly hunter
# cell further down, which reuses red/yellow/magenta for the same three kinds in its own
# separate legend.
VITALS_KIND_COLORS = {
    '518 lag detected': 'sandybrown', 'Heartbeat lag detected': 'hotpink', 'Event loop lag detected': 'orchid',
}
VITALS_KIND_ORDER = ['518 lag detected', 'Heartbeat lag detected', 'Event loop lag detected']

if len(connection_events_df):
    print(f"Connection events: {len(connection_events_df)}")
    print(connection_events_df.kind.value_counts().to_string())

    errors_df = connection_events_df[connection_events_df.error_code.notna()]
    if len(errors_df):
        print(f"\nClassified failure events: {len(errors_df)}")
        print(errors_df.error_code.value_counts().to_string())
else:
    print("No connection-lifecycle events found in this capture.")
if len(vitals_for_chart_df):
    print(f"\nVitals lag events (heartbeat/position/event-loop): {len(vitals_for_chart_df)}")
    print(vitals_for_chart_df.kind.value_counts().to_string())

# Full lifecycle timeline, not just failures: y-axis is error_code for a classified
# failure, and its bare `kind` otherwise (driver (re)starts, init start/done, WiFi
# interface selection, client-driven reconnect/config actions, and now the three
# heartbeat/position/event-loop lag warnings from load_vitals() -- early-warning symptoms
# that often precede a disconnect, per the Notes worked example below) -- analogous to the
# HIGH CPU LOAD chart above, but with no natural continuous y-value to plot, so the
# categorical axis carries the signal instead. sync_observed is excluded (high-volume,
# mostly context -- see the table below); everything else that happened is on here.
timeline_df = connection_events_df[connection_events_df.kind != 'sync_observed'].copy() \
    if len(connection_events_df) else connection_events_df
if len(timeline_df):
    timeline_df['y_label'] = timeline_df.error_code.fillna(timeline_df.kind)
    timeline_df['color_key'] = timeline_df.error_category.fillna(timeline_df.kind)

    _LIFECYCLE_HOVER = {
        'driver_start': 'Driver process (re)started',
        'init_start': 'Polaris communication init started',
        'init_done': 'Polaris communication init completed',
        'wifi_interface': 'WiFi interface selected',
    }
    def _hover_meaning(r):
        if pd.notna(r.error_meaning):
            return r.error_meaning
        if r.kind == 'config_update':
            params = {c[len('param_'):]: r[c] for c in timeline_df.columns
                      if c.startswith('param_') and pd.notna(r.get(c))}
            return 'Config change: ' + (', '.join(f'{k}={v}' for k, v in params.items()) or '(no params)')
        if r.kind == 'client_action':
            return f"Client REST action (ClientID {r.get('client')})"
        return _LIFECYCLE_HOVER.get(r.kind, '')
    timeline_df['hover_meaning'] = timeline_df.apply(_hover_meaning, axis=1)
    timeline_df['hover_detail'] = timeline_df.detail.fillna('')

if len(vitals_for_chart_df):
    vitals_chart_rows = vitals_for_chart_df.copy()
    vitals_chart_rows['y_label'] = vitals_chart_rows['kind']
    vitals_chart_rows['color_key'] = vitals_chart_rows['kind']
    vitals_chart_rows['hover_meaning'] = vitals_chart_rows.apply(
        lambda r: f"lag={r.lag_s:.2f}s -- CPU {r.cpu_pct:.0f}%  Mem {r.mem_pct:.0f}% ({r.mem_mb}MB)  Swap {r.swap_pct:.0f}%", axis=1)
    vitals_chart_rows['hover_detail'] = vitals_chart_rows.apply(
        lambda r: f"Threads {r.threads}  NetDrops {r.net_dropin}/{r.net_dropout}  NetErr {r.net_errin}/{r.net_errout}", axis=1)
    _chart_cols = ['timestamp', 'kind', 'y_label', 'color_key', 'hover_meaning', 'hover_detail']
    timeline_df = pd.concat([timeline_df[_chart_cols], vitals_chart_rows[_chart_cols]], ignore_index=True) \
        if len(timeline_df) else vitals_chart_rows[_chart_cols]

if len(timeline_df):
    label_order = [k for k in LIFECYCLE_KIND_ORDER if k in timeline_df.kind.values]
    vitals_label_order = [k for k in VITALS_KIND_ORDER if k in timeline_df.kind.values]
    error_order = (connection_events_df.loc[connection_events_df.error_code.notna(), 'error_code']
                   .value_counts().index.tolist()) if len(connection_events_df) else []
    category_array = list(reversed(label_order + vitals_label_order + error_order))  # first entry plots at the bottom

    fig = go.Figure()
    for color_key, g in timeline_df.groupby('color_key'):
        color = (ERROR_CATEGORY_COLORS.get(color_key) or LIFECYCLE_KIND_COLORS.get(color_key)
                 or VITALS_KIND_COLORS.get(color_key, 'white'))
        fig.add_trace(go.Scatter(
            x=g.timestamp, y=g.y_label, mode='markers', name=color_key,
            marker=dict(size=11, color=color, line=dict(width=1, color='black')),
            customdata=g[['kind', 'hover_detail', 'hover_meaning']],
            hovertemplate='%{x}<br><b>%{y}</b>  (kind=%{customdata[0]})<br>%{customdata[2]}'
                           '<br><i>%{customdata[1]}</i><extra></extra>',
        ))
    fig.update_layout(height=550, width=1200, template='plotly_dark',
        title='Connection lifecycle timeline -- every event + lag warning, error-coded where applicable',
        xaxis_title='Time', yaxis_title='error_code / kind',
        yaxis=dict(categoryorder='array', categoryarray=category_array),
        legend_title='category / kind', hovermode='closest')
    fig.show()
else:
    print("\nNo connection-lifecycle or vitals-lag events in this capture -- nothing to chart.")

# Full, untruncated detail/error_meaning text -- sync_observed is high-volume and mostly
# just context, so it's excluded here (still present in connection_events_df itself).
_display_cols = ['timestamp', 't_sec', 'kind', 'detail', 'error_code', 'error_meaning', 'error_category']
_param_cols = [c for c in connection_events_df.columns if c.startswith('param_')] if len(connection_events_df) else []
with pd.option_context('display.max_colwidth', None, 'display.width', 220, 'display.max_rows', None):
    display(connection_events_df.loc[connection_events_df.kind != 'sync_observed',
        [c for c in _display_cols if c in connection_events_df.columns] + _param_cols])

Connection events: 336
kind
sync_observed             205
connect_attempt_failed    108
config_update              11
init_start                  3
init_done                   3
disconnect                  3
driver_start                1
client_action               1
wifi_interface              1

Classified failure events: 111
error_code
CONNECT_TIMEOUT    108
WSAECONNRESET        3

Vitals lag events (heartbeat/position/event-loop): 170
kind
Event loop lag detected    69
Heartbeat lag detected     55
518 lag detected           46


,timestamp,t_sec,kind,detail,error_code,error_meaning,error_category,param_advanced_pec,param_pano_name,param_location,param_z3_min_limit,param_z1_min_limit,param_advanced_alignment
0,2026-09-02 17:53:01.108,0.000,driver_start,v2.2.0 Beta 4.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-09-02 17:53:08.330,7.222,connect_attempt_failed,Connect timed out. Check the Polaris is powered on and the host is joined to its Wi-Fi network.,CONNECT_TIMEOUT,"Fresh TCP connect() itself never completed within the 5s attempt window -- Polaris is off, not yet booted, or the host hasn't joined the polaris_XXXXXX hotspot yet.",connect_timeout,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-09-02 17:53:23.355,22.247,connect_attempt_failed,Connect timed out. Check the Polaris is powered on and the host is joined to its Wi-Fi network.,CONNECT_TIMEOUT,"Fresh TCP connect() itself never completed within the 5s attempt window -- Polaris is off, not yet booted, or the host hasn't joined the polaris_XXXXXX hotspot yet.",connect_timeout,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-09-02 17:53:38.408,37.300,connect_attempt_failed,Connect timed out. Check the Polaris is powered on and the host is joined to its Wi-Fi network.,CONNECT_TIMEOUT,"Fresh TCP connect() itself never completed within the 5s attempt window -- Polaris is off, not yet booted, or the host hasn't joined the polaris_XXXXXX hotspot yet.",connect_timeout,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-09-02 17:53:51.458,50.350,client_action,Polaris:bleEnableWifi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2026-09-02 17:53:53.418,52.310,connect_attempt_failed,Connect timed out. Check the Polaris is powered on and the host is joined to its Wi-Fi network.,CONNECT_TIMEOUT,"Fresh TCP connect() itself never completed within the 5s attempt window -- Polaris is off, not yet booted, or the host hasn't joined the polaris_XXXXXX hotspot yet.",connect_timeout,NaN,NaN,NaN,NaN,NaN,NaN
6,2026-09-02 17:54:03.720,62.612,wifi_interface,Wi-Fi 2 (TP-Link Wireless USB Adapter),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2026-09-02 17:54:08.471,67.363,connect_attempt_failed,Connect timed out. Check the Polaris is powered on and the host is joined to its Wi-Fi network.,CONNECT_TIMEOUT,"Fresh TCP connect() itself never completed within the 5s attempt window -- Polaris is off, not yet booted, or the host hasn't joined the polaris_XXXXXX hotspot yet.",connect_timeout,NaN,NaN,NaN,NaN,NaN,NaN
8,2026-09-02 17:54:18.539,77.431,init_start,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2026-09-02 17:54:19.274,78.166,init_done,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Driver vitals events (position-update-lag / heartbeat-lag / event-loop-lag)

Three distinct triggers sharing the same CPU/Mem/Swap/thread-count/network-counter
snapshot -- see `load_vitals()`'s docstring for what each means. `Polaris position
update` is a telemetry symptom (518 running late); `Heartbeat lag detected` and `Event
loop lag detected` are both local event-loop-stall symptoms -- a sign of CPU contention
on the machine running the driver, not necessarily a network problem at all.

In [4]:
vitals_df = load_vitals(log_filenames, log_dir=LOG_DIR)
if len(vitals_df):
    origin = kf_df.timestamp.iloc[0] if len(kf_df) else vitals_df.timestamp.iloc[0]
    vitals_df["t_sec"] = (vitals_df.timestamp - origin).dt.total_seconds()
    print(f"Vitals events: {len(vitals_df)}")
    print(vitals_df.kind.value_counts().to_string())
    print(f"\nLargest or Maxium lag (seconds) by kind:")
    print(vitals_df.groupby('kind').lag_s.max().to_string())
else:
    print("No vitals-carrying warning lines found in this capture.")
vitals_df.sort_values(by="lag_s",ascending=False).head(20)


Vitals events: 170
kind
Event loop lag detected    69
Heartbeat lag detected     55
518 lag detected           46

Largest or Maxium lag (seconds) by kind:
kind
518 lag detected           4.957
Event loop lag detected    3.641
Heartbeat lag detected     4.389


,timestamp,kind,lag_s,cpu_pct,mem_pct,mem_mb,swap_pct,threads,net_dropin,net_dropout,net_errin,net_errout,t_sec
167,2026-09-02 22:41:40.459,518 lag detected,4.957,100.0,73.9,5864,5.3,12,0,0,0,0,17317.363
33,2026-09-02 18:52:51.216,Heartbeat lag detected,4.389,100.0,74.2,5894,6.2,13,0,0,0,0,3588.120
125,2026-09-02 20:52:58.809,Event loop lag detected,3.641,71.4,75.3,5981,5.4,13,0,0,0,0,10795.713
35,2026-09-02 18:52:52.933,Event loop lag detected,3.474,100.0,72.8,5782,6.2,13,0,0,0,0,3589.837
31,2026-09-02 18:52:49.305,518 lag detected,2.892,100.0,75.4,5987,6.2,11,0,0,0,0,3586.209
165,2026-09-02 22:41:38.279,Heartbeat lag detected,2.876,100.0,73.4,5826,5.3,12,0,0,0,0,17315.183
66,2026-09-02 18:56:18.249,Event loop lag detected,2.631,100.0,75.9,6027,5.3,13,0,0,0,0,3795.153
40,2026-09-02 18:52:59.332,Event loop lag detected,2.511,100.0,71.3,5658,6.2,13,0,0,0,0,3596.236
92,2026-09-02 20:33:10.913,Event loop lag detected,2.510,100.0,70.5,5600,5.5,12,0,0,0,0,9607.817
50,2026-09-02 18:54:07.625,Event loop lag detected,2.507,100.0,78.6,6239,5.7,13,0,0,0,0,3664.529


# HIGH CPU LOAD breakdown (system resource contention)

Whenever a vitals snapshot above observes CPU > 95%, the driver triggers a one-shot
top-5-process probe (`shr.system_cpu()`), logged as its own `HIGH CPU LOAD breakdown`
line -- the single most direct signal for "was this dropout caused by something else on
the machine competing for CPU", as opposed to a genuine network/WiFi/Polaris-side
problem. An empty result here for a dropout you're investigating is itself informative --
it points *away* from local CPU contention as the cause.

In [5]:
high_cpu_df = load_high_cpu_events(log_filenames, log_dir=LOG_DIR)
if len(high_cpu_df):
    print(f"HIGH CPU LOAD events: {high_cpu_df.timestamp.nunique()} snapshot(s), {len(high_cpu_df)} process-rows")
    print("\nMost frequent top-1 (highest CPU%) process per snapshot:")
    print(high_cpu_df[high_cpu_df['rank'] == 1].process.value_counts().to_string())

    top1 = high_cpu_df[high_cpu_df['rank'] == 1]
    fig = go.Figure()
    for proc, g in top1.groupby('process'):
        fig.add_trace(go.Scatter(x=g.timestamp, y=g.cpu_pct, mode='markers', name=proc,
            marker=dict(size=6)))
    fig.update_layout(height=400, width=1200, template='plotly_dark',
        title='Top CPU-consuming process at each HIGH CPU LOAD snapshot',
        xaxis_title='Time', yaxis_title='CPU %% (per-core normalized)', hovermode='x unified')
    fig.show()
else:
    print("No HIGH CPU LOAD breakdown lines found -- no CPU-contention events observed in this capture.")
high_cpu_df.head(20)


HIGH CPU LOAD events: 16 snapshot(s), 80 process-rows

Most frequent top-1 (highest CPU%) process per snapshot:
process
python.exe      9
NINA.exe        4
WmiPrvSE.exe    2
svchost.exe     1


,timestamp,rank,process,cpu_pct
0,2026-09-02 17:58:33.494,1,NINA.exe,31.0
1,2026-09-02 17:58:33.494,2,python.exe,13.5
2,2026-09-02 17:58:33.494,3,System,6.3
3,2026-09-02 17:58:33.494,4,msedgewebview2.exe,3.6
4,2026-09-02 17:58:33.494,5,msedgewebview2.exe,2.9
5,2026-09-02 18:44:09.886,1,python.exe,13.7
6,2026-09-02 18:44:09.886,2,System,1.4
7,2026-09-02 18:44:09.886,3,NINA.exe,1.1
8,2026-09-02 18:44:09.886,4,svchost.exe,1.0
9,2026-09-02 18:44:09.886,5,WmiPrvSE.exe,0.6


# Telemetry gaps (KFLOG/PIDLOG-visible dropouts)

Per `docs/control.md`: the Polaris sometimes stops sending 518 telemetry for a few
seconds, during which `theta_pv` freezes while `theta_sp` keeps advancing. Flags any gap
much longer than the normal ~0.1-0.2s KFLOG cadence.

**This is not the same thing as a true connection outage** -- see "Outage reconstruction"
below. The driver's KF/PID control loop keeps ticking (and being logged) on dead-reckoned
predictions even after the real connection has died, so a KFLOG-visible gap can
meaningfully *undershoot* how long the mount was actually disconnected.

In [6]:
GAP_THRESHOLD_SEC = 1.0   # normal cadence is ~0.1-0.2s; a real dropout runs several seconds or more

if len(kf_df):
    gaps_df = kf_df[kf_df.gap_sec > GAP_THRESHOLD_SEC][["timestamp", "t_sec", "gap_sec"]].reset_index(drop=True)
    print(f"Flagged {len(gaps_df)} telemetry gap(s) > {GAP_THRESHOLD_SEC}s, "
          f"totalling {gaps_df.gap_sec.sum():.1f}s of KFLOG-visible lost telemetry")
else:
    gaps_df = pd.DataFrame(columns=["timestamp", "t_sec", "gap_sec"])
    print("No KF data loaded.")
gaps_df


No KF data loaded.


,timestamp,t_sec,gap_sec


# Outage reconstruction

Collapses the event stream above into discrete outage spans (`reconstruct_outages()`),
each annotated with what triggered it, how many reconnect attempts failed, whether a
driver/process restart happened mid-outage, whether a WiFi rejoin was attempted and
failed, and any config changes made while it was down. **Prefer this table's
`outage_start`/`outage_end` over the Telemetry gaps table above** when the two disagree --
see the markdown there for why the KFLOG-visible gap can undershoot the true outage.

In [7]:
outages_df = reconstruct_outages(connection_events_df)

if not len(outages_df):
    print("No outages reconstructed -- either no disconnects occurred, or load_connection_events() found nothing.")
else:
    print(f"Reconstructed {len(outages_df)} outage(s):")
    for _, o in outages_df.iterrows():
        end_str = f"{o.outage_end}" if not o.ongoing else "(still down at end of loaded logs)"
        print(f"\n  {o.outage_start}  ->  {end_str}  ({o.duration_min:.1f} min)" if not o.ongoing else
              f"\n  {o.outage_start}  ->  {end_str}")
        print(f"    trigger: {o.trigger_kind} -- {o.trigger_detail}")
        print(f"    failed reconnect attempts: {o.n_connect_attempts_failed}")
        print(f"    driver restarted mid-outage: {o.driver_restarted} (x{o.n_driver_restarts})")
        print(f"    WiFi rejoin failed: {o.wifi_join_failed}")
        if o.config_changes:
            print(f"    config changes mid-outage:")
            for cc in o.config_changes:
                print(f"      {cc['timestamp']}: {cc['params']}")

        # Cross-check against the KFLOG-visible gap (if any) covering/near this outage --
        # gaps_df's own timestamp is the *first sample after* the gap (when telemetry
        # resumed), which can land well after outage_end (a KFLOG gap resolving on its own,
        # unrelated to this outage's own reconnect) -- compare each gap's actual
        # [timestamp - gap_sec, timestamp] interval against the outage span instead of just
        # the gap's own endpoint timestamp.
        if len(gaps_df):
            oend = o.outage_end if not o.ongoing else pd.Timestamp.max
            gap_starts = gaps_df.timestamp - pd.to_timedelta(gaps_df.gap_sec, unit='s')
            overlapping = gaps_df[(gap_starts <= oend) & (gaps_df.timestamp >= o.outage_start)]
            if len(overlapping):
                kf_gap_sec = overlapping.gap_sec.sum()
                diff = kf_gap_sec - o.duration_min * 60
                relation = ('about the same as' if abs(diff) < 30 else
                            'shorter than (KFLOG kept logging dead-reckoned data through part of it)' if diff < 0 else
                            'longer than (KFLOG silence extended past reconnect -- check config_changes above, e.g. log_position left off)')
                print(f"    KFLOG-visible gap(s) overlapping this outage: {kf_gap_sec:.1f}s total, "
                      f"{relation} the {o.duration_min*60:.0f}s true outage duration")
            else:
                print(f"    No KFLOG-visible gap found near this outage -- KF/PID likely kept logging "
                      f"dead-reckoned data through it, or log_position was off already.")

        # CPU context right before the outage began
        if len(high_cpu_df):
            lookback = high_cpu_df[(high_cpu_df.timestamp >= o.outage_start - pd.Timedelta('30s')) &
                                    (high_cpu_df.timestamp <= o.outage_start)]
            if len(lookback):
                top = lookback[lookback['rank'] == 1].sort_values('timestamp')
                print(f"    CPU context in the 30s before: {', '.join(f'{r.process} {r.cpu_pct:.0f}%' for _, r in top.iterrows())}")

outages_df


Reconstructed 3 outage(s):

  2026-09-02 18:52:51.836000  ->  2026-09-02 18:56:38.509000  (3.8 min)
    trigger: disconnect -- [WinError 10054] An existing connection was forcibly closed by the remote host
    failed reconnect attempts: 14
    driver restarted mid-outage: False (x0)
    WiFi rejoin failed: False

  2026-09-02 20:33:11.973000  ->  2026-09-02 20:52:59.267000  (19.8 min)
    trigger: disconnect -- [WinError 10054] An existing connection was forcibly closed by the remote host
    failed reconnect attempts: 78
    driver restarted mid-outage: False (x0)
    WiFi rejoin failed: False

  2026-09-02 22:41:40.910000  ->  (still down at end of loaded logs)
    trigger: disconnect -- [WinError 10054] An existing connection was forcibly closed by the remote host
    failed reconnect attempts: 11
    driver restarted mid-outage: False (x0)
    WiFi rejoin failed: False


,outage_start,trigger_kind,trigger_detail,n_connect_attempts_failed,driver_restarted,n_driver_restarts,wifi_join_failed,config_changes,outage_end,ongoing,duration_min
0,2026-09-02 18:52:51.836,disconnect,[WinError 10054] An existing connection was fo...,14,False,0,False,[],2026-09-02 18:56:38.509,False,3.777883
1,2026-09-02 20:33:11.973,disconnect,[WinError 10054] An existing connection was fo...,78,False,0,False,[],2026-09-02 20:52:59.267,False,19.788233
2,2026-09-02 22:41:40.910,disconnect,[WinError 10054] An existing connection was fo...,11,False,0,False,[],NaT,True,NaN


# Comms Anomaly hunter: interactive explorer per dropout/lag event

Two sub-signatures, both surfaced in one table (`kind` column) and one dropdown/slider:
- **`dropout`** -- a hard gap, from `gaps_df` (receipt silence > `GAP_THRESHOLD_SEC`).
- **`lag`** -- 518 messages kept arriving, but consistently later than nominal, for a
  sustained run of ticks -- never long enough to register as a `gaps_df` gap, but still a
  real comms-quality problem (`measurement_lag_s` KFLOG field; requires a capture taken
  after that field was added).

For each event: driver state either side of it (`age_518`/`connected`/`mode` if present),
and position drift (`theta_state` before vs. after -- dead reckoning during a dropout can
leave the filter's belief measurably off from where the mount actually ended up).

In [8]:
COMMS_LOOKBACK_S  = 5.0                 # seconds of PID history just before an event, to check for an early-warning trend
COMMS_RECOVERY_S  = 30.0                # seconds after an event to include as resettle context
COMMS_WINDOW_PAD_S = 5.0                # extra padding either side of [t_start, t_end], for context
LAG_THRESHOLD_S   = 0.15                # measurement_lag_s (s) above which a tick counts as "running late"
LAG_MIN_RUN       = 5                   # minimum consecutive elevated ticks to count as a sustained lag event
LAG_CLUSTER_GAP_S = 2.0                 # merge elevated runs within this many seconds of each other into one event

AXIS_COLORS = {1: ("royalblue", "deepskyblue"), 2: ("mediumseagreen", "palegreen"), 3: ("orange", "navajowhite")}

def event_shapes(t_start, t_end, t_center):
    return [
        dict(type="line", xref="x", yref="paper", x0=0, x1=0, y0=0, y1=1,
             line=dict(color="red", width=1, dash="dash")),
        dict(type="rect", xref="x", yref="paper",
             x0=t_start - t_center, x1=t_end - t_center, y0=0, y1=1,
             fillcolor="red", opacity=0.08, line_width=0),
    ]

def window_slice(df, t_center, before_sec, after_sec=None):
    if after_sec is None:
        after_sec = before_sec
    w = df[(df.t_sec > t_center - before_sec) & (df.t_sec < t_center + after_sec)].copy()
    w["t_rel"] = w.t_sec - t_center
    return w

def clipped_shapes(ev, before, after):
    t_start = max(ev.t_start, ev.t_sec - before)
    t_end = min(ev.t_end, ev.t_sec + after)
    return event_shapes(t_start, t_end, ev.t_sec)

def _nearest_row(df, t_sec):
    if not len(df):
        return None
    return df.loc[(df.t_sec - t_sec).abs().idxmin()]

def _context(t_before, t_after):
    row_before = _nearest_row(pid_df, t_before)
    row_after  = _nearest_row(pid_df, t_after)
    lookback = pid_df[(pid_df.t_sec >= t_before - COMMS_LOOKBACK_S) & (pid_df.t_sec <= t_before)]
    age_518_trend = (float(lookback["age_518"].iloc[-1] - lookback["age_518"].iloc[0]))\
        if "age_518" in lookback.columns and len(lookback) >= 2 else np.nan
    return dict(
        mode_before=(row_before.get("mode") if row_before is not None else None),
        mode_after=(row_after.get("mode") if row_after is not None else None),
        age_518_before=(row_before.get("age_518") if row_before is not None else np.nan),
        age_518_trend_5s=age_518_trend,
        connected_after=(row_after.get("connected") if row_after is not None else None),
    )

comms_rows = []
theta_cols = [c for c in kf_df.columns if c.startswith("\u03b8_state_")] if len(kf_df) else []
for _, g in gaps_df.iterrows():
    t_before = g.t_sec - g.gap_sec
    t_after  = g.t_sec
    theta_before = _nearest_row(kf_df, t_before)
    theta_after  = _nearest_row(kf_df, t_after)
    if theta_cols and theta_before is not None and theta_after is not None:
        drift_arcsec = float(np.linalg.norm(
            (theta_after[theta_cols] - theta_before[theta_cols]).to_numpy(dtype=float)) * ARCSEC)
    else:
        drift_arcsec = np.nan
    t_center = (t_before + t_after) / 2
    comms_rows.append(dict(
        kind="dropout", t_sec=t_center, t_start=t_before, t_end=t_after,
        recovery_end=t_after + COMMS_RECOVERY_S,
        gap_sec=g.gap_sec, peak_lag_s=np.nan, n_ticks=np.nan, timestamp=g.timestamp,
        drift_arcsec=drift_arcsec,
        short_label=f"dropout @ {t_center:.1f}s",
        label=f"dropout @ {t_center:.1f}s ({g.gap_sec:.1f}s)",
        **_context(t_before, t_after),
    ))

lag_rows = []
if len(kf_df) and "measurement_lag_s" in kf_df.columns:
    near_gap = pd.Series(False, index=kf_df.index)
    for _, g in gaps_df.iterrows():
        near_gap |= (kf_df.t_sec >= g.t_sec - g.gap_sec - 2.0) & (kf_df.t_sec <= g.t_sec + 30.0)
    elevated = (kf_df["measurement_lag_s"] > LAG_THRESHOLD_S) & (~near_gap)
    lag_ticks = kf_df.loc[elevated, ["t_sec", "timestamp", "measurement_lag_s"]].copy()
    if len(lag_ticks):
        group = (lag_ticks.t_sec.diff().fillna(0) > LAG_CLUSTER_GAP_S).cumsum()
        for _, run in lag_ticks.assign(group=group).groupby("group"):
            if len(run) < LAG_MIN_RUN:
                continue
            peak = run.loc[run.measurement_lag_s.idxmax()]
            t_start, t_end = run.t_sec.min(), run.t_sec.max()
            lag_rows.append(dict(
                kind="lag", t_sec=peak.t_sec, t_start=t_start, t_end=t_end,
                recovery_end=t_end + COMMS_RECOVERY_S,
                gap_sec=np.nan, peak_lag_s=peak.measurement_lag_s, n_ticks=len(run), timestamp=peak.timestamp,
                drift_arcsec=np.nan,
                short_label=f"lag @ {peak.t_sec:.1f}s",
                label=f"lag @ {peak.t_sec:.1f}s ({len(run)} ticks, peak {peak.measurement_lag_s:.2f}s)",
                **_context(t_start, t_end),
            ))
else:
    print("Note: measurement_lag_s not present in this capture's KFLOG -- lag sub-detection skipped.")

comms_events_df = pd.DataFrame(comms_rows + lag_rows)
if len(comms_events_df):
    comms_events_df = comms_events_df.sort_values("t_sec").reset_index(drop=True)
n_dropout = int((comms_events_df.kind == "dropout").sum()) if len(comms_events_df) else 0
n_lag     = int((comms_events_df.kind == "lag").sum())     if len(comms_events_df) else 0
print(f"Characterized {len(comms_events_df)} Comms Anomaly event(s): {n_dropout} hard dropout(s) (> {GAP_THRESHOLD_SEC}s), "
      f"{n_lag} sustained-lag event(s) ({LAG_THRESHOLD_S:.2f}s+ for >= {LAG_MIN_RUN} ticks).")
comms_events_df


Note: measurement_lag_s not present in this capture's KFLOG -- lag sub-detection skipped.
Characterized 0 Comms Anomaly event(s): 0 hard dropout(s) (> 1.0s), 0 sustained-lag event(s) (0.15s+ for >= 5 ticks).


""


In [9]:
fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
    specs=[[{}], [{"secondary_y": True}], [{}], [{}]],
    subplot_titles=["518 timing: receipt interval vs measurement_lag_s (s)",
                     "driver vitals events: lag_s (dots) -- CPU/Mem/Swap % hidden, own axis",
                     "\u03b8 error (arcsec) -- all axes", "\u03c9_op (arcsec/s) -- all axes"],
    row_heights=[0.2, 0.3, 0.25, 0.25], vertical_spacing=0.06)

if not len(comms_events_df):
    print("No Comms Anomaly events -- nothing to hunt.")
else:
    events = comms_events_df.reset_index(drop=True)
    trace_spans = []
    hidden_trace_indices = []
    VITALS_KIND_COLORS = {"518 lag detected": "yellow", "Heartbeat lag detected": "red", "Event loop lag detected": "magenta"}
    for ev_i, ev in events.iterrows():
        visible = (ev_i == 0)
        start = len(fig.data)
        before = ev.t_sec - ev.t_start + COMMS_WINDOW_PAD_S
        after  = ev.recovery_end - ev.t_sec + COMMS_WINDOW_PAD_S

        kf_window = window_slice(kf_df, ev.t_sec, before, after) if len(kf_df) else pd.DataFrame()
        t_rel_kf = kf_window.t_rel if len(kf_window) else pd.Series(dtype=float)
        fig.add_trace(go.Scatter(x=t_rel_kf, y=(kf_window.gap_sec if len(kf_window) else []), name="518 receipt", visible=visible,
            mode="markers+lines", marker=dict(size=4, color="yellow"), line=dict(color="yellow", width=1)), row=1, col=1)
        lag_col = kf_window["measurement_lag_s"] if len(kf_window) and "measurement_lag_s" in kf_window.columns else pd.Series(dtype=float)
        fig.add_trace(go.Scatter(x=t_rel_kf, y=lag_col, name="measurement_lag_s", visible=visible,
            mode="markers+lines", marker=dict(size=4, color="orange"), line=dict(color="orange", width=1)), row=1, col=1)

        vitals_window = vitals_df[(vitals_df.t_sec > ev.t_sec - before) & (vitals_df.t_sec < ev.t_sec + after)].copy() \
            if len(vitals_df) else pd.DataFrame()
        if len(vitals_window):
            vitals_window["t_rel"] = vitals_window.t_sec - ev.t_sec
        for kind, color in VITALS_KIND_COLORS.items():
            sub = vitals_window[vitals_window.kind == kind] if len(vitals_window) else pd.DataFrame()
            hover = (sub.apply(lambda r: f"lag={r.lag_s:.2f}s  CPU={r.cpu_pct:.0f}% Mem={r.mem_pct:.0f}% Swap={r.swap_pct:.0f}%  "
                                          f"Threads={r.threads} NetDrops={r.net_dropin}/{r.net_dropout} NetErr={r.net_errin}/{r.net_errout}", axis=1)
                     if len(sub) else pd.Series(dtype=object))
            fig.add_trace(go.Scatter(x=(sub.t_rel if len(sub) else []), y=(sub.lag_s if len(sub) else []),
                name=kind, visible=visible, mode="markers", marker=dict(size=9, color=color, symbol="circle",
                line=dict(width=1, color="black")), text=hover, hovertemplate="%{text}<extra>" + kind + "</extra>"),
                row=2, col=1, secondary_y=False)
        hidden_idx = []
        for metric, color in [("mem_pct", "deepskyblue"), ("cpu_pct", "palegreen"), ("swap_pct", "orange")]:
            hidden_idx.append(len(fig.data))
            fig.add_trace(go.Scatter(x=(vitals_window.t_rel if len(vitals_window) else []),
                y=(vitals_window[metric] if len(vitals_window) else []),
                name=metric, visible=False, mode="markers+lines", marker=dict(size=4, color=color),
                line=dict(color=color, width=1, dash="dot")), row=2, col=1, secondary_y=True)
        hidden_trace_indices.append(hidden_idx)

        pid_window = window_slice(pid_df, ev.t_sec, before, after)
        t_rel = pid_window.t_rel
        for ax_idx in (1, 2, 3):
            meas_color, _ = AXIS_COLORS[ax_idx]
            error = (wrap_deg(pid_window[f"\u03b8_pv_{ax_idx}"] - pid_window[f"\u03b8_sp_{ax_idx}"]) * ARCSEC)
            fig.add_trace(go.Scatter(x=t_rel, y=error, name=f"M{ax_idx} error", visible=visible,
                line=dict(color=meas_color, width=1)), row=3, col=1)
            omega_op = pid_window[f"\u03c9_op_{ax_idx}"] * ARCSEC
            fig.add_trace(go.Scatter(x=t_rel, y=omega_op, name=f"M{ax_idx} \u03c9_op", visible=visible,
                line=dict(color=meas_color, width=1)), row=4, col=1)
        trace_spans.append((start, len(fig.data) - start))

    n_total = len(fig.data)
    buttons = []
    for ev_i, ev in events.iterrows():
        start, count = trace_spans[ev_i]
        visible = [False] * n_total
        for j in range(start, start + count):
            visible[j] = True
        for idx in hidden_trace_indices[ev_i]:
            visible[idx] = False
        buttons.append(dict(
            label=ev.short_label, method="update",
            args=[{"visible": visible},
                  {"title": f"Comms Anomaly: {ev.label}",
                   "shapes": clipped_shapes(ev, ev.t_sec - ev.t_start + COMMS_WINDOW_PAD_S,
                                             ev.recovery_end - ev.t_sec + COMMS_WINDOW_PAD_S)}],
        ))

    slider_steps = list(buttons)

    fig.update_yaxes(title_text="lag (s)", row=2, col=1, secondary_y=False)
    fig.update_yaxes(title_text="CPU/Mem/Swap %", row=2, col=1, secondary_y=True)
    fig.update_xaxes(title_text="Time relative to event (s)", row=4, col=1)
    fig.update_layout(height=1100, width=1300, template="plotly_dark", hovermode="x unified",
        title=f"Comms Anomaly: {events.iloc[0].label}",
        shapes=clipped_shapes(events.iloc[0],
                               events.iloc[0].t_sec - events.iloc[0].t_start + COMMS_WINDOW_PAD_S,
                               events.iloc[0].recovery_end - events.iloc[0].t_sec + COMMS_WINDOW_PAD_S),
        legend=dict(groupclick="toggleitem"),
        updatemenus=[dict(buttons=buttons, direction="down", x=1.0, xanchor="right", y=1.1, yanchor="top",
                           showactive=True, bgcolor="#2B2B2B", bordercolor="#777777", borderwidth=1,
                           font=dict(color="#EEEEEE"))],
        sliders=[dict(active=0, x=0.0, len=0.88, pad=dict(t=60, b=10),
                       currentvalue=dict(prefix="Event (use \u2190/\u2192 or drag to step): ", font=dict(color="#EEEEEE", size=12)),
                       font=dict(color="#EEEEEE", size=10), bgcolor="#2B2B2B", bordercolor="#777777",
                       activebgcolor="#555555", steps=slider_steps)])
    fig.show()


No Comms Anomaly events -- nothing to hunt.


# Notes

**Worked example: the 2026-09-01T00:21:52 dropout in `alpaca.soak_nopec_Beta4.3_08_31a12.log`.**
Reconstructed above as a single ~29.8-minute outage (00:21:52.445 -> 00:51:43.158), not
the ~15.5-minute KFLOG-visible gap (00:41:42 -> 00:57:12) `docs/pec_theta_space_plan.md`
originally reported -- that gap turned out to be caused by a client toggling
`log_position` off mid-outage (00:41:43.836) and back on after reconnecting (00:57:12.543),
not the true start/end of the connection loss.

Root cause chain, read directly off the events above:
1. `HIGH CPU LOAD breakdown` at 00:21:59 shows NINA.exe (38.5%) + python.exe (23.4%, the
   driver's own process) pushing the host CPU near 100% -- corroborated by escalating
   `Heartbeat lag detected`/`Event loop lag detected` vitals events in the preceding
   ~10 seconds (pulse lag climbing 0.447s -> 5.955s; event-loop sleep overrun up to
   3.237s against a 500ms budget).
2. That starved the driver's asyncio event loop long enough that the socket wasn't
   serviced in time -- at 00:21:52.445, `==DISCONNECT== Polaris socket error: [WinError
   10054] An existing connection was forcibly closed by the remote host`. **Not** the
   driver's own 5s-no-518 watchdog (`age_518` was only ~1.1s at the moment of the
   position-update-lag warning immediately before this) -- the remote side (or the OS/
   network stack under load) reset the connection first.
3. 110 `Connect timed out` reconnect attempts followed, all failing, for ~25 minutes.
4. A WiFi rejoin was attempted via `Polaris:bleEnableWifi`/`Polaris:ConnectPolaris`
   REST actions at 00:41:12-00:41:29 and failed (`Failed to join WiFi network
   'polaris_b83c06': netsh wlan connect failed: ... error 0x57`).
5. A fresh driver process started at 00:47:07 (`==STARTUP==`) -- the miniPC/driver
   restart -- but still couldn't connect for another ~4.5 minutes.
6. `Polaris:ConnectPolaris` at 00:51:42 finally succeeded, `Polaris communication
   init... done` at 00:51:43.158, closing the outage.

See `docs/pec_theta_space_plan.md` for how this correction affects that session's
tracking-segment boundaries.